# 課題と解答例：31_train_boarding_ca

元Notebook: [../31_train_boarding_ca.ipynb](../31_train_boarding_ca.ipynb)

## 10. 課題（実装）

1. `door_width_sweep(door_halves, seeds)` を作り，複数のドア幅と複数seedで停車時間を計算し，平均，標準偏差，最大値をDataFrameにまとめる．
2. `passenger_count_sweep(counts, seeds)` を作り，乗車人数だけを変えたときの停車時間分布を表にする．隣り合う人数での平均停車時間の差分も列に加える．
3. ドア幅または人数を変えた結果について，停車時間の平均だけでなく，密度図または打ち切りの有無も確認するコードを追加する．
4. 停車時間が線形に増えているか急増しているかを，作成した表の差分列から説明する．

## 解答例

1. ドア幅スイープの例である．

   ```python
   def door_width_sweep(door_halves, seeds):
       rows = []
       for dh in door_halves:
           for seed in seeds:
               t, *_ = simulate(H, W, DOOR_COL, dh, N_ALIGHT, N_BOARD, standers=0, max_steps=MAX_STEPS, seed=seed)
               rows.append({"door_half": dh, "seed": seed, "dwell_time": t})
       raw = pd.DataFrame(rows)
       return raw.groupby("door_half")["dwell_time"].agg(["mean", "std", "max"]).reset_index()
   ```

2. 人数スイープでは差分列を追加する．

   ```python
   def passenger_count_sweep(counts, seeds):
       rows = []
       for n in counts:
           for seed in seeds:
               t, *_ = simulate(H, W, DOOR_COL, DOOR_HALF, N_ALIGHT, n, standers=0, max_steps=MAX_STEPS, seed=seed)
               rows.append({"n_board": n, "seed": seed, "dwell_time": t})
       summary = pd.DataFrame(rows).groupby("n_board")["dwell_time"].mean().reset_index()
       summary["mean_diff"] = summary["dwell_time"].diff()
       return summary
   ```

3. 密度図は，平均停車時間が大きい条件を選び，`simulate` が返す密度配列を `imshow` で描く．打ち切りを扱う場合は，`t >= MAX_STEPS` を `cutoff` として保存する．

4. `mean_diff` がほぼ一定なら線形増加に近い．人数が増えるほど `mean_diff` も増えるなら，ドア付近の混雑による急増が起きている可能性がある．